# Transformação Bronze → Silver

**Propósito:** Orquestrador de produção da transformação Bronze → Silver. Itera as 11
tabelas de evento definidas em `CONFIGURACAO_TABELAS`
(`src/transformacao/configuracao_tabelas.py`) e aplica `transformar_bronze_para_silver`
(ADR-013) em cada uma, gravando via `MERGE INTO` por chave de negócio.

**Tipo:** Notebook orquestrador de produção — Task do `job_diario`, roda diariamente
às 06:00 via Asset Bundle.

**Posição na cadeia:** depende de `ingerir_dados` ter concluído (Bronze precisa existir).
Roda em paralelo com `promover_seeds` — os dois alimentam `construir_gold`, que depende
das duas Tasks.

**Contexto:** criado para corrigir uma lacuna real de produção — a transformação
existia como código testado desde o ADR-013, mas nunca foi migrada para uma Task
do Asset Bundle após a migração para `mode: production`. Era chamada apenas em
notebooks de spike/backfill manual, deixando a Silver órfã desde 21/08/2026 sem
nenhum erro visível (Job Notifications nunca disparou, porque as Tasks existentes
continuavam retornando `sucesso`).

**Registro de observabilidade:** cada tabela processada é registrada individualmente
em `pipeline_runs` via `registrar_execucao`, seguindo o mesmo padrão dos outros
6 orquestradores.

**Autor:** Bruno Queles
**Data de criação:** 2026-09-15

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

if "/files/" in notebook_path:
    root_relative = notebook_path.split("/files/")[0] + "/files"
else:
    root_relative = notebook_path.rsplit("/src/", 1)[0]

candidatos = {root_relative}
if root_relative.startswith("/Workspace"):
    candidatos.add(root_relative[len("/Workspace"):])
else:
    candidatos.add("/Workspace" + root_relative)

for candidato in candidatos:
    if candidato not in sys.path:
        sys.path.append(candidato)

print("Candidatos adicionados ao sys.path:", candidatos)

In [0]:
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver
from src.observabilidade.registrar_execucao import registrar_execucao_pipeline

CATALOG = "poc_pulse_observability"
PIPELINE = "transformar_silver"

resultados = []

for tabela, config in CONFIGURACAO_TABELAS.items():
    try:
        resultado = transformar_bronze_para_silver(spark, CATALOG, tabela, config)
        registrar_execucao_pipeline(
            spark=spark,
            pipeline=PIPELINE,
            item=tabela,
            status=resultado["status"],
            detalhes=resultado,
        )
        resultados.append(resultado)
        print(f"OK  {tabela}: {resultado['status']} ({resultado['total_linhas']} linhas)")
    except Exception as e:
        registrar_execucao_pipeline(
            spark=spark,
            pipeline=PIPELINE,
            item=tabela,
            status="falha",
            detalhes={"erro": str(e)},
        )
        resultados.append({"tabela": tabela, "status": "falha", "erro": str(e)})
        print(f"FALHOU {tabela}: {e}")